# Portugal – Sensitivity Analysis: Solar PV Capital Costs (`03_sensitivity_analysis.ipynb`)

Builds on outputs from `01_potenzialanalyse.ipynb` (CSV capacity factors and region files).

**Assignment option 2:** Successively reduce the capital cost of solar PV from 100% down to 0% in 25% steps under the net-zero CO₂ constraint. Describe how the optimised system changes.

**Requires (from 01_potenzialanalyse.ipynb):**
- `portugal_regions.gpkg`
- `p_max_pu_solar.csv`, `p_max_pu_onshore.csv`, `p_max_pu_offshore.csv`
- `p_nom_max.csv`

**Requires (downloaded automatically):**
- `global_power_plant_database.csv`
- `load.csv`
- PyPSA technology-data costs (2030, via GitHub)

In [ ]:
import pypsa
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlretrieve
from itertools import combinations
from pypsa.costs import annuity
import os

resolution = 3  # hours (3h temporal resolution)
print(f"PyPSA version: {pypsa.__version__}")

## 1. Regions and cost data

In [ ]:
regions = gpd.read_file("portugal_regions.gpkg")

bevoelkerung = {
    "Norte": 3_587_074,
    "Centro": 2_227_567,
    "Lisboa": 2_870_770,
    "Alentejo": 704_707,
    "Algarve": 467_475,
}
regions["bevoelkerung"] = regions["region"].map(bevoelkerung)
regions["bev_anteil"] = regions["bevoelkerung"] / regions["bevoelkerung"].sum()
print(regions[["region", "x", "y", "bev_anteil"]])

In [ ]:
year = 2030
url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{year}.csv"
costs = pd.read_csv(url, index_col=[0, 1])
costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs.unit = costs.unit.str.replace("/kW", "/MW")
defaults = {"FOM": 0, "VOM": 0, "efficiency": 1, "fuel": 0, "investment": 0,
            "lifetime": 25, "CO2 intensity": 0, "discount rate": 0.07}
costs = costs.value.unstack().fillna(defaults)
costs.at["OCGT", "fuel"] = costs.at["gas", "fuel"]
costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]
costs.at["OCGT", "CO2 intensity"] = costs.at["gas", "CO2 intensity"]
costs.at["CCGT", "CO2 intensity"] = costs.at["gas", "CO2 intensity"]
costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
annuity_factor = annuity(costs["discount rate"], costs["lifetime"])
costs["capital_cost"] = (annuity_factor + costs["FOM"] / 100) * costs["investment"]
print(f"Cost data loaded: {year}, {len(costs)} technologies")

## 2. Existing power plants (from GPPD)

In [ ]:
fn_gppd = "global_power_plant_database.csv"
if not os.path.exists(fn_gppd):
    print("Downloading GPPD...")
    urlretrieve(
        "https://tubcloud.tu-berlin.de/s/567ckizz2Y6RLQq/download?path=%2Fglobal-power-plant-database&files=global_power_plant_database.csv",
        fn_gppd,
    )
gppd = pd.read_csv(fn_gppd, low_memory=False)

pt_plants = gppd[(gppd["country"] == "PRT") &
                  (~gppd["primary_fuel"].isin(["Wind", "Solar"]))].copy()
punkte = gpd.GeoDataFrame(
    pt_plants,
    geometry=gpd.points_from_xy(pt_plants["longitude"], pt_plants["latitude"]),
    crs=4326,
)
punkte = gpd.sjoin(punkte, regions[["region", "geometry"]], how="left", predicate="within")
pt_plants = punkte[punkte["region"].notna()].copy()

pt_plants["generation_gwh"] = (
    pt_plants["generation_gwh_2013"].fillna(pt_plants["estimated_generation_gwh_2013"])
)

fuel_to_tech = {"Gas": "CCGT", "Coal": "coal", "Biomass": "biomass", "Waste": "waste CHP"}
thermal = pt_plants[pt_plants["primary_fuel"].isin(fuel_to_tech)].copy()
thermal["tech"] = thermal["primary_fuel"].map(fuel_to_tech)
agg_thermal = thermal.groupby(["region", "tech"]).agg(
    p_nom=("capacity_mw", "sum")
).reset_index()

hydro = pt_plants[pt_plants["primary_fuel"] == "Hydro"].copy()
agg_hydro = hydro.groupby("region").agg(
    p_nom=("capacity_mw", "sum"), erzeugung_gwh=("generation_gwh", "sum")
).reset_index()
agg_hydro["p_max_pu"] = (agg_hydro["erzeugung_gwh"] * 1e3) / (agg_hydro["p_nom"] * 8760)

print(f"Thermal generators: {len(agg_thermal)}, Hydro: {len(agg_hydro)}")

## 3. Load time series

In [ ]:
fn_load = "load.csv"
if not os.path.exists(fn_load):
    print("Downloading load data...")
    urlretrieve(
        "https://tubcloud.tu-berlin.de/s/567ckizz2Y6RLQq/download?path=%2Fgegis&files=load.csv",
        fn_load,
    )
load_raw = pd.read_csv(fn_load, usecols=["time", "PT"])
load_raw["time"] = pd.to_datetime(load_raw["time"])
load_raw = load_raw.set_index("time")["PT"]
print(f"Load data: {load_raw.index[0]} to {load_raw.index[-1]}, mean={load_raw.mean():.0f} MW")

## 4. Renewable capacity factors and potentials

In [ ]:
snapshots = pd.date_range("2013-01-01", "2014-01-01", freq=f"{resolution}h", inclusive="left")

p_max_pu_solar = pd.read_csv("p_max_pu_solar.csv", index_col=0, parse_dates=True)
p_max_pu_onshore = pd.read_csv("p_max_pu_onshore.csv", index_col=0, parse_dates=True)
p_max_pu_offshore = pd.read_csv("p_max_pu_offshore.csv", index_col=0, parse_dates=True)
p_nom_max = pd.read_csv("p_nom_max.csv", index_col=0)

p_max_pu_solar = p_max_pu_solar.resample(f"{resolution}h").mean().reindex(snapshots)
p_max_pu_onshore = p_max_pu_onshore.resample(f"{resolution}h").mean().reindex(snapshots)
p_max_pu_offshore = p_max_pu_offshore.resample(f"{resolution}h").mean().reindex(snapshots)

print("p_nom_max [MW]:"); print(p_nom_max.round(0))

## 5. Transmission distances

In [ ]:
regions_m = regions.set_index("region").to_crs(3035)
nachbarn = [
    (a, b) for a, b in combinations(regions_m.index, 2)
    if regions_m.loc[a, "geometry"].buffer(500).intersects(regions_m.loc[b, "geometry"].buffer(500))
]

punkte_geo = gpd.GeoSeries(
    gpd.points_from_xy(regions["x"], regions["y"]), index=regions["region"], crs=regions.crs
).to_crs("EPSG:4087")
lufblinie_km = pd.DataFrame(
    {a: {b: punkte_geo[a].distance(punkte_geo[b]) / 1e3 for b in punkte_geo.index} for a in punkte_geo.index}
)
print("Neighboring pairs:", nachbarn)

## 6. `build_network()` function

Rebuilds a complete PyPSA network from scratch. The `solar_cost_factor` scales the solar PV capital cost (1.0 = 100%, 0.0 = free).

In [ ]:
def build_network(solar_cost_factor=1.0):
    """Full PyPSA network with scaled solar capital cost."""
    net = pypsa.Network()
    pypsa.options.params.optimize.include_objective_constant = True

    net.add("Bus", regions["region"], x=regions["x"].values, y=regions["y"].values, carrier="AC")
    net.set_snapshots(snapshots)
    net.snapshot_weightings.loc[:, :] = resolution

    alle_carrier = ["CCGT", "coal", "biomass", "waste CHP", "hydro",
                    "solar", "onwind", "offwind-float",
                    "battery storage", "hydrogen storage underground"]
    co2 = [costs.at[c, "CO2 intensity"] if c in costs.index else 0 for c in alle_carrier]
    net.add("Carrier", alle_carrier, co2_emissions=co2)

    # Existing conventional fleet (fixed, not extendable)
    for _, row in agg_thermal.iterrows():
        net.add("Generator", f"{row['region']} {row['tech']}", bus=row["region"],
                carrier=row["tech"], p_nom=row["p_nom"], p_nom_extendable=False,
                marginal_cost=costs.at[row["tech"], "marginal_cost"],
                capital_cost=costs.at[row["tech"], "capital_cost"],
                efficiency=costs.at[row["tech"], "efficiency"])
    for _, row in agg_hydro.iterrows():
        net.add("Generator", f"{row['region']} hydro", bus=row["region"],
                carrier="hydro", p_nom=row["p_nom"], p_nom_extendable=False,
                p_max_pu=row["p_max_pu"], marginal_cost=costs.at["hydro", "marginal_cost"])

    # Load (population-weighted)
    load_pt = load_raw.resample(f"{resolution}h").mean().reindex(net.snapshots)
    for _, row in regions.iterrows():
        net.add("Load", row["region"], bus=row["region"], p_set=load_pt * row["bev_anteil"])

    # Renewable generators (extendable)
    solar_cc = costs.at["solar", "capital_cost"] * solar_cost_factor
    tech_map = {
        "solar":         (p_max_pu_solar,    solar_cc),
        "onwind":        (p_max_pu_onshore,  costs.at["onwind", "capital_cost"]),
        "offwind-float": (p_max_pu_offshore, costs.at["offwind-float", "capital_cost"]),
    }
    for tech, (pmax_df, cap_cost) in tech_map.items():
        for region in regions["region"]:
            if region not in pmax_df.columns or region not in p_nom_max.index:
                continue
            net.add("Generator", f"{region} {tech}", bus=region, carrier=tech,
                    p_nom=0, p_nom_extendable=True,
                    p_nom_max=p_nom_max.at[region, tech],
                    p_max_pu=pmax_df[region],
                    capital_cost=cap_cost,
                    marginal_cost=costs.at[tech, "marginal_cost"])

    # Transmission links (700 EUR/MW/km x 1.5 x crow-fly distance)
    for a, b in nachbarn:
        laenge_km = luftblinie_km.at[a, b] * 1.5
        net.add("Link", f"{a} - {b}", bus0=a, bus1=b, p_min_pu=-1,
                p_nom=0, p_nom_extendable=True, efficiency=1,
                capital_cost=700 * laenge_km, carrier="AC")

    # Battery storage (2h, 4h, 6h per region)
    for region in regions["region"]:
        for h in [2, 4, 6]:
            net.add("StorageUnit", f"{region} battery {h}h", bus=region,
                    carrier="battery storage", max_hours=h, p_nom=0, p_nom_extendable=True,
                    capital_cost=(costs.at["battery inverter", "capital_cost"]
                                  + h * costs.at["battery storage", "capital_cost"]),
                    efficiency_store=costs.at["battery inverter", "efficiency"],
                    efficiency_dispatch=costs.at["battery inverter", "efficiency"])

    # Hydrogen storage (168h, 336h, 672h per region)
    for region in regions["region"]:
        for h in [168, 336, 672]:
            net.add("StorageUnit", f"{region} hydrogen {h}h", bus=region,
                    carrier="hydrogen storage underground", max_hours=h, p_nom=0, p_nom_extendable=True,
                    capital_cost=(costs.at["electrolysis", "capital_cost"]
                                  + costs.at["fuel cell", "capital_cost"]
                                  + h * costs.at["hydrogen storage underground", "capital_cost"]),
                    efficiency_store=costs.at["electrolysis", "efficiency"],
                    efficiency_dispatch=costs.at["fuel cell", "efficiency"])
    return net

print(f"build_network() defined. Base solar capital cost: {costs.at['solar', 'capital_cost']:.0f} EUR/MW/a")

## 7. Sensitivity loop

Run 5 optimisations with solar PV capital cost at 100%, 75%, 50%, 25%, 0% of the 2030 base value. Each run enforces the net-zero CO₂ constraint.

In [ ]:
solar_factors = [1.0, 0.75, 0.50, 0.25, 0.0]
results = []

for factor in solar_factors:
    label = f"{int(factor * 100)}%"
    print(f"\nSolving: solar cost = {label} ...", flush=True)
    net = build_network(solar_cost_factor=factor)
    net.add("GlobalConstraint", "CO2Limit", carrier_attribute="co2_emissions", sense="<=", constant=0)
    net.optimize(solver_name="highs", log_to_console=False)

    caps = net.generators.groupby("carrier")["p_nom_opt"].sum()
    stor = net.storage_units.groupby("carrier")["p_nom_opt"].sum()
    gen = (net.generators_t.p.multiply(net.snapshot_weightings.generators, axis=0)
               .groupby(net.generators.carrier, axis=1).sum().sum() / 1e6)  # TWh

    results.append({
        "Solar cost": label,
        "System cost [bn EUR/a]": net.objective / 1e9,
        "Solar [GW]": caps.get("solar", 0) / 1e3,
        "Onshore wind [GW]": caps.get("onwind", 0) / 1e3,
        "Offshore wind [GW]": caps.get("offwind-float", 0) / 1e3,
        "CCGT [GW]": caps.get("CCGT", 0) / 1e3,
        "Battery [GW]": stor.get("battery storage", 0) / 1e3,
        "Hydrogen [GW]": stor.get("hydrogen storage underground", 0) / 1e3,
        "Transmission [GW]": net.links["p_nom_opt"].sum() / 1e3,
        "Solar gen [TWh]": gen.get("solar", 0),
        "Onshore gen [TWh]": gen.get("onwind", 0),
        "Offshore gen [TWh]": gen.get("offwind-float", 0),
    })
    print(f"  Cost: {net.objective/1e9:.2f} bn EUR/a | Solar: {caps.get('solar', 0)/1e3:.1f} GW")

df = pd.DataFrame(results).set_index("Solar cost")
print("\n=== RESULTS ===")
print(df.round(2))

In [ ]:
df.round(3).to_csv("sensitivity_results.csv")
print("Saved: sensitivity_results.csv")

## 8. Visualisation

In [ ]:
x = df.index.tolist()
colors = {"Solar [GW]": "#f4a261", "Onshore wind [GW]": "#457b9d",
          "Offshore wind [GW]": "#1d3557", "CCGT [GW]": "#8d99ae"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: System cost
axes[0].plot(x, df["System cost [bn EUR/a]"], marker="o", linewidth=2, color="steelblue")
for xi, yi in zip(x, df["System cost [bn EUR/a]"]):
    axes[0].annotate(f"{yi:.2f}", (xi, yi), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
axes[0].set_title("Total System Cost", fontsize=12, fontweight="bold")
axes[0].set_ylabel("bn EUR/year")
axes[0].set_xlabel("Solar PV capital cost")
axes[0].grid(True, alpha=0.3)

# Panel 2: Installed capacity (stacked bar)
bottom = np.zeros(len(x))
for col, color in colors.items():
    vals = df[col].values
    axes[1].bar(x, vals, bottom=bottom, label=col.replace(" [GW]", ""), color=color, alpha=0.9)
    bottom += vals
axes[1].set_title("Installed Capacity (Extendable)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("GW")
axes[1].set_xlabel("Solar PV capital cost")
axes[1].legend(fontsize=8, loc="upper right")
axes[1].grid(True, alpha=0.3, axis="y")

# Panel 3: Storage
axes[2].plot(x, df["Battery [GW]"], marker="s", label="Battery", color="#2a9d8f", linewidth=2)
axes[2].plot(x, df["Hydrogen [GW]"], marker="^", label="Hydrogen", color="#e76f51", linewidth=2)
axes[2].plot(x, df["Transmission [GW]"], marker="D", label="Transmission", color="#6c757d", linewidth=2)
axes[2].set_title("Storage & Transmission Capacity", fontsize=12, fontweight="bold")
axes[2].set_ylabel("GW")
axes[2].set_xlabel("Solar PV capital cost")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle("Sensitivity Analysis: Solar PV Capital Cost (Net-Zero CO\u2082)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("sensitivity_solar_cost.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sensitivity_solar_cost.png")

In [ ]:
# Generation mix as percentage
gen_cols = ["Solar gen [TWh]", "Onshore gen [TWh]", "Offshore gen [TWh]"]
gen_df = df[gen_cols].copy()
gen_df.columns = ["Solar PV", "Onshore Wind", "Offshore Wind"]

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(x))
for col, color in zip(gen_df.columns, ["#f4a261", "#457b9d", "#1d3557"]):
    ax.bar(x, gen_df[col].values, bottom=bottom, label=col, color=color, alpha=0.9)
    bottom += gen_df[col].values
ax.set_title("Annual Renewable Generation Mix by Solar Cost Scenario", fontsize=12, fontweight="bold")
ax.set_ylabel("TWh/year")
ax.set_xlabel("Solar PV capital cost")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("sensitivity_generation_mix.png", dpi=150, bbox_inches="tight")
plt.show()